# ARC Task Annotator

Browse tasks one at a time. For each task you see:
- All training pairs (input → output grids)
- Computed structural features

Type your description of the rule, then **Save & Next**.
Descriptions are saved to `data/human_descriptions.json` after each save.

**Controls:** Save & Next · Skip · ← Prev · jump to any task ID via the text box.

In [ ]:
%matplotlib inline
import json, sys
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output

def _find_project_root() -> Path:
    p = Path.cwd()
    for _ in range(8):
        if (p / 'data' / 'training').exists():
            return p
        if (p / 'CLAUDE.md').exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        'Cannot find project root. '
        'Run: jupyter notebook from inside the arc-agi-solver directory.'
    )

PROJECT  = _find_project_root()
DATA_DIR = PROJECT / 'data' / 'training'
DESC_FILE = PROJECT / 'data' / 'human_descriptions.json'

assert DATA_DIR.exists(), f"Training data not found at {DATA_DIR}"

# ARC colour palette
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

# Load all tasks sorted by ID
task_files = sorted(DATA_DIR.glob('*.json'))
task_ids   = [f.stem for f in task_files]
tasks      = {f.stem: json.loads(f.read_text()) for f in task_files}
print(f"Loaded {len(tasks)} tasks")

# Load existing descriptions
if DESC_FILE.exists():
    human_descs = json.loads(DESC_FILE.read_text())
    print(f"Found {len(human_descs)} existing descriptions")
else:
    human_descs = {}
    print("No existing descriptions — starting fresh")

In [ ]:
def compute_features(task):
    """Compute programmatic structural features from training pairs."""
    pairs = task['train']
    in_shapes  = [(np.array(p['input']).shape)  for p in pairs]
    out_shapes = [(np.array(p['output']).shape) for p in pairs]

    # Size relationship
    ratios = []
    size_descs = []
    for (ih, iw), (oh, ow) in zip(in_shapes, out_shapes):
        if (oh, ow) == (ih, iw):
            size_descs.append('same')
        elif oh == ih * 3 and ow == iw * 3:
            size_descs.append('3× scale')
        elif oh == ih * 2 and ow == iw * 2:
            size_descs.append('2× scale')
        elif oh < ih or ow < iw:
            size_descs.append(f'shrink → {oh}×{ow}')
        else:
            size_descs.append(f'{ih}×{iw} → {oh}×{ow}')

    size_summary = size_descs[0] if len(set(size_descs)) == 1 else '  |  '.join(size_descs)

    # Output always fixed same size?
    fixed_out = len(set(out_shapes)) == 1
    fixed_out_size = out_shapes[0] if fixed_out else None

    # Colour counts
    in_colors  = [set(np.array(p['input']).flat)  for p in pairs]
    out_colors = [set(np.array(p['output']).flat) for p in pairs]
    in_counts  = [len(c) for c in in_colors]
    out_counts = [len(c) for c in out_colors]

    # Colours always in every input / output
    always_in  = set.intersection(*in_colors)  - {0}
    always_out = set.intersection(*out_colors) - {0}

    # Output colours always subset of input colours?
    out_subset = all(oc <= ic for oc, ic in zip(out_colors, in_colors))
    new_colors = not out_subset

    lines = [
        f"Size:          {size_summary}",
        f"Output fixed:  {'yes — ' + str(fixed_out_size[0]) + '×' + str(fixed_out_size[1]) if fixed_out else 'no (varies)'}",
        f"Colors in:     {in_counts} unique per pair  (always present: {sorted(always_in) or 'none'})",
        f"Colors out:    {out_counts} unique per pair  (always present: {sorted(always_out) or 'none'})",
        f"New colors:    {'yes — output introduces colors not in input' if new_colors else 'no — output only uses input colors'}",
        f"Train pairs:   {len(pairs)}",
    ]
    return '\n'.join(lines)

In [ ]:
class TaskAnnotator:

    def __init__(self):
        # Start at first unannotated task
        self.idx = next(
            (i for i, tid in enumerate(task_ids) if tid not in human_descs),
            0
        )

        # ── Widgets ──────────────────────────────────────────────────────
        self.fig_out  = widgets.Output()
        self.feat_out = widgets.Output()

        self.textarea = widgets.Textarea(
            placeholder='Describe the transformation rule in your own words…',
            layout=widgets.Layout(width='100%', height='130px')
        )
        self.progress  = widgets.HTML()
        self.status    = widgets.HTML()

        self.btn_prev  = widgets.Button(description='← Prev',
                                        layout=widgets.Layout(width='100px'))
        self.btn_skip  = widgets.Button(description='Skip',
                                        layout=widgets.Layout(width='100px'))
        self.btn_save  = widgets.Button(description='Save & Next',
                                        button_style='success',
                                        layout=widgets.Layout(width='130px'))

        self.jump_box  = widgets.Text(placeholder='task id…',
                                      layout=widgets.Layout(width='160px'))
        self.btn_jump  = widgets.Button(description='Go',
                                        layout=widgets.Layout(width='60px'))

        self.btn_prev.on_click(lambda _: self.move(-1))
        self.btn_skip.on_click(lambda _: self.move(+1))
        self.btn_save.on_click(lambda _: self.save_and_next())
        self.btn_jump.on_click(lambda _: self.jump())

        # ── Layout ───────────────────────────────────────────────────────
        controls = widgets.HBox([
            self.btn_prev, self.btn_skip, self.btn_save,
            widgets.Label('  Jump to:'), self.jump_box, self.btn_jump
        ])
        ui = widgets.VBox([
            self.progress,
            self.fig_out,
            self.feat_out,
            widgets.HTML('<b>Your description:</b>'),
            self.textarea,
            controls,
            self.status,
        ])
        display(ui)
        self.render()

    # ── Rendering ─────────────────────────────────────────────────────────

    def render(self):
        tid  = task_ids[self.idx]
        task = tasks[tid]
        done = len(human_descs)
        annotated_marker = ' ✓' if tid in human_descs else ''

        self.progress.value = (
            f'<b>Task {self.idx + 1} / {len(task_ids)}</b> — '
            f'<code>{tid}</code>{annotated_marker} &nbsp;·&nbsp; '
            f'{done} / {len(task_ids)} annotated'
        )

        # Pre-fill textarea if already annotated
        self.textarea.value = human_descs.get(tid, {}).get('description', '')
        self.status.value = ''

        # Grid figure
        pairs = task['train']
        n = len(pairs)
        with self.fig_out:
            clear_output(wait=True)
            fig, axes = plt.subplots(n, 2, figsize=(6, 2.5 * n), squeeze=False)
            for i, pair in enumerate(pairs):
                for j, (grid, label) in enumerate([
                    (np.array(pair['input']),  f'Train {i+1} input'),
                    (np.array(pair['output']), f'Train {i+1} output'),
                ]):
                    axes[i, j].imshow(grid, cmap=CMAP, norm=NORM,
                                      interpolation='nearest')
                    axes[i, j].set_title(label, fontsize=9)
                    axes[i, j].axis('off')
            plt.tight_layout()
            plt.show()

        # Features
        with self.feat_out:
            clear_output(wait=True)
            feats = compute_features(task)
            print(feats)

    # ── Actions ───────────────────────────────────────────────────────────

    def save_and_next(self):
        tid  = task_ids[self.idx]
        desc = self.textarea.value.strip()
        if not desc:
            self.status.value = '<span style="color:orange">⚠ Description is empty — skipping save.</span>'
            self.move(+1)
            return
        human_descs[tid] = {
            'description': desc,
            'timestamp':   datetime.now().isoformat(timespec='seconds'),
        }
        DESC_FILE.write_text(json.dumps(human_descs, indent=2))
        self.status.value = f'<span style="color:green">✓ Saved {tid}</span>'
        self.move(+1)

    def move(self, delta):
        self.idx = max(0, min(len(task_ids) - 1, self.idx + delta))
        self.render()

    def jump(self):
        target = self.jump_box.value.strip()
        if target in task_ids:
            self.idx = task_ids.index(target)
            self.render()
        else:
            self.status.value = f'<span style="color:red">Task {target!r} not found</span>'


annotator = TaskAnnotator()